In [ ]:
import os
import subprocess
import pandas as pd


# Load the uploaded CSV file to examine its contents
file_path = 'cache_adopted_repo.csv'
repo_data = pd.read_csv(file_path)

# Display the first few rows to understand its structure
repo_data.head()
# Load the data
repo_names = repo_data['Repo name'].unique()

# Base directory for storing outputs
output_base_dir = './outputs'
os.makedirs(output_base_dir, exist_ok=True)

# List to store failed repositories
failed_repos = []

for i, repo_name in enumerate(repo_names):
    print(f"\nProcessing repository {i + 1}/{len(repo_names)}: {repo_name}")

    # Generate URL and paths
    repo_url = f"https://github.com/{repo_name}.git"
    repo_dir_name = repo_name.replace('/', '_')
    output_csv_path = os.path.join(output_base_dir, f"{repo_dir_name}_workflows.csv")
    workflows_dir = os.path.join(output_base_dir, repo_dir_name)
    
    os.makedirs(workflows_dir, exist_ok=True)
    
    # Construct gigawork command
    command = f"gigawork {repo_url} -n {repo_dir_name} -o {output_csv_path} -w {workflows_dir} -sa"
    
    print(f"\nRunning command:\n{command}\n")
    
    # Execute the command and capture the output
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    # Stream the output in real-time
    for line in process.stdout:
        print(line, end='')  # Print stdout in real-time

    for line in process.stderr:
        print(f"ERROR: {line}", end='')  # Print stderr in real-time

    process.wait()  # Wait for process to complete
    
    if process.returncode == 0:
        print(f"\nSuccessfully processed {repo_name}.\n")
    else:
        print(f"\nFailed to process {repo_name}.\n")
        failed_repos.append(repo_name)  # Add to failed repos list

# Save failed repositories to a CSV file
failed_repos_df = pd.DataFrame(failed_repos, columns=['Repo name'])
failed_csv_path = os.path.join(output_base_dir, "failed_repos.csv")
failed_repos_df.to_csv(failed_csv_path, index=False)

print(f"\nList of failed repositories saved to: {failed_csv_path}\n")
